# Track 08 — Memory Engineering for RAG & Autonomous Agents
### Mastering Multi-Tier Memory: Short-Term Buffers, Semantic Episodic Vectors, Entity Graphs, and Ebbinghaus Consolidation

In naive Retrieval-Augmented Generation, each user query is treated as an isolated, stateless event. However, real-world conversational agents and multi-turn enterprise systems require **statefulness**: remembering user preferences, referencing past decisions, avoiding repetitive explanations, and resolving temporal contradictions (e.g., *"I moved from London to Berlin"*).

Merely stuffing all past conversation turns into the LLM context window causes **context window pollution**, quadratic inference cost escalation, attentional degradation (*lost-in-the-middle* phenomenon), and critical **privacy leakage** when sensitive data is indiscriminately retained.

```
       Naive Stateless RAG                          Production Memory-Engineered RAG
   ┌────────────────────────┐                   ┌──────────────────────────────────────┐
   │ User Query:            │                   │ User Query:                          │
   │ "Where do I live?"     │                   │ "Where do I live?"                   │
   │           ▼            │                   │                  ▼                   │
   │ Vector DB (Docs only)  │                   │ Privacy Guardrail -> Redact Secrets  │
   │           ▼            │                   │                  ▼                   │
   │ Response:              │                   │ Multi-Tier Memory Retrieval:         │
   │ "I have no knowledge   │                   │ - Active Facts: (User, location, Berlin)
   │  of your residence."   │                   │ - Episodic Vectors: past relocation  │
   │                        │                   │ - Working Scratchpad: active goals   │
   │                        │                   │                  ▼                   │
   │                        │                   │ Response:                            │
   │                        │                   │ "You currently reside in Berlin,     │
   │                        │                   │  having relocated from London."      │
   └────────────────────────┘                   └──────────────────────────────────────┘
```

---

### Multi-Tier Memory Architecture

Production memory engineering organizes memory across four distinct operational tiers:

1. **Tier 1: Working Memory Scratchpad** — Ephemeral state tracking current user goals, active constraints, and execution variables.
2. **Tier 2: Short-Term Dialogue Buffers** — Bounded token-sliding buffers and running summary accumulators.
3. **Tier 3: Entity Knowledge Graph / Fact Store** — Deterministic Entity-Attribute-Value triples with automatic revision pointers and conflict resolution.
4. **Tier 4: Long-Term Semantic Episodic Memory** — Multi-tenant vector store with **Ebbinghaus forgetting curve decay** and spaced-repetition reinforcement.

### Architectural Topology

The following diagram illustrates the complete dataflow across all memory tiers, from raw user input through automated PII sanitization, extraction, consolidation, and augmented prompt synthesis.

In [ ]:
from IPython.display import Image, display

display(Image("assets/diagrams/track08_multi_tier_memory_architecture.png"))

## 1. Setup & Environment Verification

We import our first-principles memory primitives from `common.memory_engineering` and configure our deterministic offline environment. All foundational memory operations run 100% offline on standard CPU without external network dependencies.

In [ ]:
import os
import time
from common.memory_engineering import (
    ChatMessage,
    SlidingWindowMemory,
    TokenBufferMemory,
    SummaryBufferMemory,
    WorkingMemoryScratchpad,
    PrivacyFilter,
    EpisodicMemoryRecord,
    SemanticEpisodicStore,
    EntityFact,
    EntityFactStore,
    FactExtractor,
    MemoryConsolidator,
    MemoryManager,
    MemoryEvaluator,
    estimate_tokens,
)
from common.config import settings

print("Memory engineering modules loaded successfully.")
print(f"LLM Provider: {settings.llm_provider}")
print(f"Groq API Key Configured: {bool(settings.groq_api_key)}")

## 2. Short-Term Memory Buffers: Token Bounds & Summary Compression

### The Context Window Dilemma
If an application retains all raw conversational turns, token consumption grows linearly $O(T)$ while transformer self-attention computation scales quadratically $O(T^2)$. Once the model context window is exceeded, the API call fails or silently truncates critical initial instructions.

To mitigate this, we implement two primary short-term memory strategies:
1. **`TokenBufferMemory`**: Enforces a strict upper bound on token consumption while **pinning** system prompt instructions so they are never evicted.
2. **`SummaryBufferMemory`**: Preserves the most recent $K$ interaction turns verbatim, while progressively condensing older evicted turns into an evolving running summary.

In [ ]:
# Demonstrate TokenBufferMemory with pinned system prompt
token_mem = TokenBufferMemory(max_tokens=60)
token_mem.add_message("system", "You are an enterprise AI architecture assistant.")

print("Adding successive dialogue turns under strict 60-token budget:")
for turn_idx in range(1, 6):
    token_mem.add_message("user", f"Turn {turn_idx}: How should we structure microservice {turn_idx}?")
    token_mem.add_message("assistant", f"Microservice {turn_idx} should use clean architecture and gRPC interfaces.")

print(f"Total tokens in buffer: {token_mem.total_tokens()} (Budget <= 60)")
print("Current Retained Buffer:")
print("-" * 50)
print(token_mem.get_formatted_history())
print("-" * 50)

### Progressive Summary Buffers
When users engage in lengthy dialogues (e.g., 20+ turns), pure eviction causes total amnesia of earlier decisions. `SummaryBufferMemory` solves this by continuously synthesizing evicted turns into a compact running synopsis:

In [ ]:
summary_mem = SummaryBufferMemory(recent_window_size=2)
summary_mem.add_message("system", "You are a DevOps migration assistant.")

dialogue = [
    ("user", "We are planning to migrate our PostgreSQL cluster to AWS Aurora."),
    ("assistant", "Aurora PostgreSQL provides managed failover and auto-scaling storage."),
    ("user", "We must maintain zero downtime during replication."),
    ("assistant", "Use AWS Database Migration Service (DMS) with continuous CDC replication."),
    ("user", "What network configuration is required?"),
    ("assistant", "Configure VPC peering and secure security group ingress on port 5432."),
]

for role, content in dialogue:
    summary_mem.add_message(role, content)

print("Augmented Context Rendered by SummaryBufferMemory:")
print("=" * 60)
print(summary_mem.get_context_for_prompt())
print("=" * 60)

## 3. Working Memory Scratchpads

Autonomous RAG systems (such as ReAct loops and Plan-and-Solve agents) require dedicated **working memory** to track active hypotheses, intermediate tool outputs, execution constraints, and unfulfilled prerequisites. 

`WorkingMemoryScratchpad` decouples ephemeral execution state from dialogue history:

In [ ]:
scratchpad = WorkingMemoryScratchpad()
scratchpad.set_goal("Deploy multi-region ChromaDB vector cluster")
scratchpad.add_constraint("Tenant data must remain encrypted at rest (AES-256)")
scratchpad.add_constraint("Maximum p99 retrieval latency < 50ms")
scratchpad.set_variable("active_primary_region", "us-east-1")
scratchpad.set_variable("read_replica_regions", ["eu-central-1", "ap-southeast-1"])

print(scratchpad.format_scratchpad())

## 4. Privacy Filter & PII Guardrails

### Why Privacy Filtering is Mandatory in RAG Memory
Long-term memory vector stores persist across sessions and days. If a user pastes an API secret, social security number, corporate email, or credit card into chat, and that string is vector-embedded into long-term memory, it creates an irreversible compliance and data-leakage risk (GDPR Article 17 *Right to be Forgotten*).

Our `PrivacyFilter` automatically detects and redacts sensitive PII before any turn is indexed into episodic or entity memory:

In [ ]:
privacy_engine = PrivacyFilter()

raw_user_input = (
    "Please update my contact record: my work email is dev.lead@fintech-corp.com, "
    "my direct line is 415-555-0199, and my personal Groq key is gsk_99xKlmZ0192837465abcedf1234. "
    "My corporate SSN is 000-12-3456."
)

sanitized_text, redactions = privacy_engine.sanitize(raw_user_input)

print("Raw User Input:")
print(raw_user_input)
print("\nSanitized Text for Storage:")
print(sanitized_text)
print(f"\nDetected & Neutralized Sensitive Entities: {redactions}")

## 5. Long-Term Semantic Episodic Memory & The Ebbinghaus Forgetting Curve

### Mathematical Foundation: Cognitive Memory Retention
Hermann Ebbinghaus discovered in 1885 that human memory decays exponentially over time without reinforcement. In production AI memory, we model this using an **effective decay rate** $\lambda_{\text{eff}}$ tempered by the **spacing effect** (access frequency $N_{\text{access}}$):

$$\Delta t = t_{\text{current}} - t_{\text{last\_accessed}}$$

$$\lambda_{\text{eff}} = \frac{\lambda_{\text{base}}}{1 + \ln(1 + N_{\text{access}})}$$

$$S_{\text{recency}} = \exp\left(-\lambda_{\text{eff}} \cdot \frac{\Delta t}{t_{1/2}}\right)$$

Where:
- $t_{1/2}$ is the configurable half-life (e.g. 24 hours).
- $N_{\text{access}}$ represents the number of times this specific memory was retrieved and accessed.
- Every recall event flattens the forgetting slope, ensuring critical knowledge persists while trivial transient context naturally fades.

### Visualizing the Ebbinghaus Decay & Spaced Repetition

Let us inspect the retention curves across multiple recall frequencies:

In [ ]:
display(Image("assets/diagrams/track08_ebbinghaus_decay_curve.png"))

### Hybrid Memory Scoring Formula
When a user submits a query $\mathbf{q}$, each stored candidate memory $\mathbf{m}$ is evaluated via a composite scoring function:

$$S_{\text{total}}(\mathbf{q}, \mathbf{m}) = \alpha \cdot \text{CosineSimilarity}(\mathbf{q}, \mathbf{m}) + \beta \cdot \text{Importance}(\mathbf{m}) + \gamma \cdot S_{\text{recency}}(\mathbf{m}, t)$$

Let us verify this behavior in practice with `SemanticEpisodicStore`:

In [ ]:
# Initialize episodic store with a 24-hour half-life
episodic_store = SemanticEpisodicStore(half_life_seconds=86400.0)

base_timestamp = 1700000000.0  # T0

# Ingest episodic interactions
m1 = episodic_store.add_memory(
    user_id="user_charlie",
    content="User configured PyTorch distributed training with DDP across 8 A100 GPUs",
    importance_score=0.9,
    timestamp=base_timestamp,
    category="event",
)

m2 = episodic_store.add_memory(
    user_id="user_charlie",
    content="User mentioned they were drinking coffee at a cafe in downtown Seattle",
    importance_score=0.2,
    timestamp=base_timestamp,
    category="event",
)

# Simulate retrieval 48 hours later (T0 + 48 hours)
t_48h = base_timestamp + (48 * 3600.0)

query = "What GPU infrastructure setup does the user use?"
results = episodic_store.retrieve(
    query=query,
    user_id="user_charlie",
    top_k=2,
    alpha_sim=0.50,
    beta_imp=0.25,
    gamma_rec=0.25,
    current_time=t_48h,
)

print("Episodic Retrieval Outcomes after 48 hours:")
for rank, res in enumerate(results, start=1):
    rec = res.record
    print(f"Rank {rank}: Total Score = {res.score:.3f} | Sim: {res.similarity_score:.3f} | Imp: {res.importance_score:.2f} | Recency: {res.recency_score:.3f}")
    print(f"  Content: {rec.content}\n")

## 6. Entity Knowledge Graphs & Temporal Conflict Resolution

### Resolving Stale Knowledge Contradictions
A frequent failure mode of naive vector retrieval is **stale memory contradiction**. If a user previously stated *"I live in Paris"* and later states *"I just relocated to Berlin"*, vector search may retrieve both passages with near-equal cosine similarity, causing the LLM to hallucinate or generate confused answers.

Our `EntityFactStore` resolves this deterministically by structuring facts as Entity-Attribute-Value tuples. When a new fact is recorded for an existing `(entity, attribute)` key, the older fact is **superseded** and marked with a tombstone pointer:

In [ ]:
fact_store = EntityFactStore()

# Step 1: Initial user location recorded at T0
f1, superseded_1 = fact_store.upsert_fact(
    user_id="user_elena",
    entity="User",
    attribute="location",
    value="Paris",
    timestamp=100.0,
)
print(f"Recorded Initial Fact: {f1.format_triple()} (Active={f1.is_active})")

# Step 2: User relocates at T1
f2, superseded_2 = fact_store.upsert_fact(
    user_id="user_elena",
    entity="User",
    attribute="location",
    value="Berlin",
    timestamp=200.0,
)
print(f"Recorded Updated Fact: {f2.format_triple()} (Active={f2.is_active})")
print(f"Superseded Old Fact ID: {superseded_2.fact_id} -> superseded_by={superseded_2.superseded_by}")

print("\nActive Entity Knowledge Graph for Elena:")
print(fact_store.format_knowledge_graph("user_elena"))

## 7. Memory Consolidation, Deduplication & Decay Pruning

### Managing Storage Lifecycle
Without active memory management, episodic databases accumulate duplicate records and transient noise. `MemoryConsolidator` executes two critical maintenance procedures:
1. **Semantic Deduplication**: Identifies semantically redundant memories using vector cosine similarity, merging access counts and preserving highest importance scores.
2. **Decay Pruning**: Periodically purges records whose composite retention drops below a threshold ($S_{\text{retention}} < 0.15$), safeguarding memory capacity.

In [ ]:
consolidator = MemoryConsolidator()

# Demonstrate Semantic Deduplication
records = [
    EpisodicMemoryRecord(
        memory_id="m101", user_id="u", session_id="s",
        content="User prefers using PostgreSQL for relational transactional data",
        timestamp=100.0, importance_score=0.5,
    ),
    EpisodicMemoryRecord(
        memory_id="m102", user_id="u", session_id="s",
        content="User prefers using PostgreSQL for relational transactional data",
        timestamp=150.0, importance_score=0.9,
    ),
]

deduped = consolidator.deduplicate(records, similarity_threshold=0.88)
print(f"Records before deduplication: {len(records)}")
print(f"Records after deduplication:  {len(deduped)}")
print(f"Merged Record Importance:     {deduped[0].importance_score} (Inherited max)")

## 8. Unified Memory Coordination & Context Augmentation

The `MemoryManager` unifies all memory tiers into an orchestrated pipeline:
1. Incoming turns are filtered for sensitive PII.
2. Short-term buffers are updated.
3. Structured facts are extracted and conflict-checked.
4. Episodic memories are indexed.
5. Queries trigger an augmented prompt assembly that presents facts, episodic context, working goals, and recent dialogue in an optimal layout for the LLM.

In [ ]:
manager = MemoryManager(max_buffer_tokens=250, half_life_seconds=86400.0)

# Ingest Turn 1
t1_res = manager.process_turn(
    user_id="engineer_42",
    session_id="dev_session",
    user_message="I prefer FastAPI with async SQLAlchemy. My email is alex@company.internal.",
    assistant_message="Understood! FastAPI and async SQLAlchemy make a great high-throughput stack.",
)

# Ingest Turn 2
t2_res = manager.process_turn(
    user_id="engineer_42",
    session_id="dev_session",
    user_message="I work as a Staff Backend Architect.",
    assistant_message="Noted! I will tailor architecture discussions to Staff-level depth.",
)

# Set working scratchpad goal
manager.scratchpad.set_goal("Design async event-driven database service")
manager.scratchpad.add_constraint("P99 latency < 25ms")

# Build Augmented Prompt for a new query
augmented_prompt = manager.build_augmented_prompt(
    user_id="engineer_42",
    session_id="dev_session",
    query="What database connection pooling strategy do you recommend for our project?",
    top_k_episodic=2,
)

print("Synthesized Augmented Prompt Context:")
print("=" * 70)
print(augmented_prompt)
print("=" * 70)

## 9. Memory Evaluation & Diagnostic Harness

We evaluate memory health using three quantitative indicators:
1. **Multi-Tenant Isolation**: Proves mathematical separation between distinct users.
2. **Contradiction Rate**: Verifies that active facts contain zero unresolved conflicts.
3. **Retrieval Precision**: Validates recall of ground-truth preferences.

In [ ]:
# 1. Multi-Tenant Isolation Audit
isolation_passed = MemoryEvaluator.evaluate_multi_tenant_isolation(
    manager.episodic_store,
    user_a="engineer_42",
    user_b="unauthorized_guest",
)

# 2. Contradiction Rate
active_facts = manager.fact_store.get_active_facts("engineer_42")
contradiction_rate = MemoryEvaluator.evaluate_contradiction_rate(active_facts)

print("=== Memory Diagnostic Audit ===")
print(f"Multi-Tenant Isolation Integrity: {'PASSED [SECURE]' if isolation_passed else 'FAILED [LEAKAGE]'}")
print(f"Knowledge Graph Contradiction Rate: {contradiction_rate * 100:.1f}%")
print(f"Active Structured Facts Count:      {len(active_facts)}")

> [!NOTE]
> ### 10. Optional Groq LLM Demonstration
> The following cell is optional. It requires a valid `GROQ_API_KEY` in `.env` and consumes API quota. If no key is configured, it falls back to a deterministic offline generation without error.

In [ ]:
if settings.groq_api_key:
    from groq import Groq
    
    print("Executing optional Groq API memory-augmented synthesis...")
    client = Groq(api_key=settings.groq_api_key)
    
    completion = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a senior AI coding assistant. Answer concisely using provided user facts."},
            {"role": "user", "content": augmented_prompt},
        ],
        temperature=0.2,
        max_tokens=250,
    )
    print("\nGroq LLM Response:")
    print("-" * 50)
    print(completion.choices[0].message.content)
    print("-" * 50)
else:
    print("GROQ_API_KEY not configured. Deterministic offline fallback output:")
    print("-" * 50)
    print("Assistant: Based on your preference for FastAPI and async SQLAlchemy as a Staff Backend Architect,")
    print("I recommend utilizing asyncpg with SQLAlchemy's AsyncEngine and QueuePool configured with a pool_size")
    print("of 20 connections and max_overflow of 10 to ensure p99 latencies stay well below 25ms.")
    print("-" * 50)

## Summary & Curriculum Progression

In Track 08, we established a rigorous, multi-tier memory architecture for production RAG and agent systems:
1. **Short-Term Context Management**: Solved token explosion and attentional decay with `TokenBufferMemory` and `SummaryBufferMemory`.
2. **Privacy Filtering**: Safeguarded long-term storage by automatically redacting PII and sensitive API keys.
3. **Ebbinghaus Forgetting Decay**: Mathematically modeled human memory decay and spaced repetition in `SemanticEpisodicStore`.
4. **Entity Knowledge Graphs**: Structured user facts as entity-attribute-value triples with automated conflict resolution.
5. **Memory Consolidation**: Implemented semantic deduplication and decay-based pruning.

**Next Track**: In **Track 09 — GraphRAG and Knowledge Graphs**, we expand beyond individual entity triples into enterprise knowledge graphs, community summaries, Cypher queries, and hybrid graph-vector retrieval.